# 6.2 常見片段怎麼合併？


兩份文件abab、abac，a-b共三次、b-a兩次、a-c一次。先把文件內的相鄰a-b各合為ab，兩份分別剩兩項與三項，原文仍是同樣四字母。

這種反覆數相鄰對、合最高頻對的方法叫BPE。玩具版從字元開始；圖中每組獨立合併，不跨文件。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-06-02-bpe-merge.svg")))

In [ ]:
from collections import Counter

corpus = [list("abab"), list("abac")]
pairs = Counter()
for row in corpus:
    for left, right in zip(row, row[1:]):
        pairs[(left, right)] += 1
pair = pairs.most_common(1)[0][0]


def merge(row, pair):
    out = []
    i = 0
    while i < len(row):
        if i + 1 < len(row) and tuple(row[i : i + 2]) == pair:
            out.append("".join(pair))
            i += 2
        else:
            out.append(row[i])
            i += 1
    return out


print(pairs)
print("選擇", pair, "結果", [merge(row, pair) for row in corpus])

輸入兩張字元清單。`row[1:]` 比原列晚一格，zip配出相鄰對，Counter數它們；`most_common(1)` 找最高頻那一項，再取其中的符號對。這裡a-b唯一最多，平手時則需規定穩定的選擇順序。計數與切片背景見 [1.4](https://birdhackor.github.io/tiny-perceptron-vlm/1.4.html)、[1.3](https://birdhackor.github.io/tiny-perceptron-vlm/1.3.html)。

`merge` 從左到右掃描。`while` 在條件成立時重複縮排代碼，i是現在位置；若還有下一項且兩項正是pair，就接成ab並跳兩格，否則複製當前項並只前進一格。`and` 要兩條件同時成立，先檢查不越界；`tuple` 把切片改成固定次序的小組，便於與pair比較。輸出四字的第一份縮成兩token，第二份縮成三token。

ab現在一token，未必是有語言意義的詞。完整BPE加新項、重新計數並保存合併順序；平手也需穩定規則。訂拆分表和訓練語言模型不同，只用訓練側訂規則，再用未參與的文字查長度與還原。這個例子只合併，沒有改字形。

練習再加一份 `list("acacacac")`，先預測a-c新增四次，總共五次，會勝過a-b的三次，再執行核對。觀察兩份原文怎麼按新pair合併，而不是只看新增文件：tokenizer規則一變，同一舊文的編碼也會跟著變。

<details>
<summary>補充：實作約定與原始紀錄</summary>

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
